# CatVTON LoRA fine-tune on VITON-HD

**What this trains:** a LoRA adapter on top of CatVTON's UNet so try-on quality jumps for clothing styles seen in VITON-HD.

**Free-tier setup:** Kaggle P100 (16GB) or T4 ×2. ~6-12 hours for 1 epoch on full VITON-HD, faster on a subset.

**Output:** a `lora_catvton/` folder (~20MB) you upload to your backend and point `VTON_LORA_CHECKPOINT` at.

## Before running
1. **GPU:** Kaggle → Settings → Accelerator → GPU P100 (or T4 ×2)
2. **Persistence:** Settings → Persistence → Files and Variables (so checkpoints survive disconnects)
3. **Internet:** Settings → Internet → On (needed for HF downloads)
4. **Dataset:** add Kaggle dataset `marquis03/high-resolution-viton-zalando-dataset` (search 'VITON-HD' in Add Data). If a different VITON-HD mirror is available, paths in `DATA_ROOT` may need adjusting — the notebook auto-detects common layouts.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade pip
!pip install -q diffusers==0.27.2 transformers==4.40.0 accelerate==0.30.0 peft==0.10.0 \
                bitsandbytes==0.43.1 safetensors==0.4.3 \
                Pillow opencv-python-headless tqdm wandb

## 2. Imports + config

In [ ]:
import os, glob, math, random, json
from pathlib import Path
from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
print(f'Device: {device} | dtype: {dtype}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
@dataclass
class Config:
    # Data
    data_root:     str = '/kaggle/input'   # auto-discovered below
    image_size:    int = 384               # VITON-HD native is 1024x768; 384 keeps Kaggle VRAM safe
    train_subset:  int = 2000              # use a subset to fit free-tier time. 0 = all (~11k)
    
    # Model
    base_model:    str = 'zheng-chong/CatVTON'
    lora_rank:     int = 16
    lora_alpha:    int = 32
    lora_dropout:  float = 0.05
    target_modules = ['to_q', 'to_k', 'to_v', 'to_out.0']  # attention only — keeps params small
    
    # Training
    batch_size:    int = 2                 # P100 16GB handles 2 @ 384, T4 might need 1
    grad_accum:    int = 4                 # effective batch = 8
    epochs:        int = 1
    lr:            float = 1e-4
    warmup_steps:  int = 100
    max_grad_norm: float = 1.0
    
    # Logging
    save_every:    int = 500
    output_dir:    str = '/kaggle/working/lora_catvton'
    seed:          int = 42

cfg = Config()
torch.manual_seed(cfg.seed); random.seed(cfg.seed); np.random.seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
print(json.dumps({k: str(v) for k, v in cfg.__dict__.items() if not k.startswith('_')}, indent=2))

## 3. Discover VITON-HD layout

Different Kaggle mirrors use different folder structures. We probe common ones.

In [ ]:
def find_vitonhd_root() -> Path:
    """Walk /kaggle/input looking for the canonical VITON-HD structure:
       <root>/train/image/*.jpg  +  <root>/train/cloth/*.jpg
       (filenames match between the two folders — that's the pair)"""
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        p = Path(root)
        if p.name == 'image' and (p.parent / 'cloth').exists():
            candidates.append(p.parent)
        if p.name == 'train' and (p / 'image').exists() and (p / 'cloth').exists():
            candidates.append(p)
    if not candidates:
        raise RuntimeError(
            'VITON-HD not found in /kaggle/input. Add the dataset via '
            'Add Data → search "VITON-HD" or "zalando viton".'
        )
    # Prefer the deepest match with both image+cloth dirs
    root = sorted(candidates, key=lambda p: len(p.parts), reverse=True)[0]
    print(f'VITON-HD root: {root}')
    print(f'  train/image: {len(list((root/"image").glob("*.jpg")))} files'
          if (root/'image').exists() else f'  (image dir not at root, looking deeper)')
    return root

VITON_ROOT = find_vitonhd_root()

## 4. Dataset

VITON-HD provides for each sample:
- `image/`: person wearing the target garment (this is the **ground truth**)
- `cloth/`: the target garment, flat
- `agnostic-v3.2/` (or `agnostic/`): person with garment region masked out (this is the **input**)
- `image-parse-v3/`: clothing-area segmentation mask

Training pair: (`agnostic` + `cloth`) → `image`. Loss = denoising loss on the masked region.

In [ ]:
class VITONHDDataset(Dataset):
    def __init__(self, root: Path, image_size: int, subset: int = 0):
        self.root = root
        self.size = image_size
        
        img_dir   = root / 'image'
        cloth_dir = root / 'cloth'
        # Agnostic dir name varies between mirrors
        agnostic_dir = None
        for candidate in ['agnostic-v3.2', 'agnostic', 'agnostic_v3.2', 'image-agnostic']:
            if (root / candidate).exists():
                agnostic_dir = root / candidate
                break
        # Mask dir
        mask_dir = None
        for candidate in ['image-parse-v3', 'image-parse', 'cloth-mask']:
            if (root / candidate).exists():
                mask_dir = root / candidate
                break
        
        self.has_agnostic = agnostic_dir is not None
        self.has_mask     = mask_dir is not None
        self.agnostic_dir = agnostic_dir
        self.mask_dir     = mask_dir
        
        # Build pair list: filenames present in BOTH image/ and cloth/
        img_names = {p.name for p in img_dir.glob('*.jpg')}
        cloth_names = {p.name for p in cloth_dir.glob('*.jpg')}
        common = sorted(img_names & cloth_names)
        if subset > 0:
            common = common[:subset]
        
        self.pairs = [(img_dir / n, cloth_dir / n) for n in common]
        print(f'Dataset: {len(self.pairs)} pairs | agnostic: {self.has_agnostic} | mask: {self.has_mask}')
    
    def __len__(self):
        return len(self.pairs)
    
    def _load_img(self, path: Path) -> torch.Tensor:
        img = Image.open(path).convert('RGB').resize((self.size, self.size), Image.LANCZOS)
        arr = np.array(img, dtype=np.float32) / 127.5 - 1.0  # [-1, 1]
        return torch.from_numpy(arr).permute(2, 0, 1)  # CHW
    
    def _load_mask(self, path: Path) -> torch.Tensor:
        m = Image.open(path).convert('L').resize((self.size, self.size), Image.NEAREST)
        arr = np.array(m, dtype=np.float32) / 255.0
        # If multi-class parse, keep upper-body label (5 in VITON-HD parse)
        if arr.max() > 1.5:
            arr = ((arr >= 4) & (arr <= 7)).astype(np.float32)
        return torch.from_numpy(arr).unsqueeze(0)  # 1HW
    
    def __getitem__(self, idx):
        img_path, cloth_path = self.pairs[idx]
        name = img_path.name
        
        person = self._load_img(img_path)
        cloth  = self._load_img(cloth_path)
        
        # Agnostic = person with garment area inpainted. If missing, fall back
        # to building a synthetic agnostic by graying out the upper torso band.
        if self.has_agnostic:
            ap = self.agnostic_dir / name
            if ap.exists():
                agnostic = self._load_img(ap)
            else:
                agnostic = self._make_synthetic_agnostic(person)
        else:
            agnostic = self._make_synthetic_agnostic(person)
        
        # Mask: where the garment goes
        if self.has_mask:
            mp = self.mask_dir / name.replace('.jpg', '.png')
            if not mp.exists():
                mp = self.mask_dir / name
            if mp.exists():
                mask = self._load_mask(mp)
            else:
                mask = self._make_synthetic_mask()
        else:
            mask = self._make_synthetic_mask()
        
        return {
            'person':   person,    # ground-truth output
            'agnostic': agnostic,  # input (person with garment area masked)
            'cloth':    cloth,     # garment reference
            'mask':     mask,      # 1HW, 1=garment area
        }
    
    def _make_synthetic_agnostic(self, person: torch.Tensor) -> torch.Tensor:
        """Gray out the upper-torso band as a coarse agnostic substitute."""
        out = person.clone()
        s = self.size
        y0, y1 = int(s * 0.20), int(s * 0.70)
        out[:, y0:y1, :] = 0.0  # neutral gray in [-1,1] space
        return out
    
    def _make_synthetic_mask(self) -> torch.Tensor:
        s = self.size
        m = torch.zeros(1, s, s)
        m[:, int(s*0.20):int(s*0.70), int(s*0.10):int(s*0.90)] = 1.0
        return m

train_ds = VITONHDDataset(VITON_ROOT, cfg.image_size, subset=cfg.train_subset)
train_dl = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                      num_workers=2, pin_memory=True, drop_last=True)
print(f'Steps per epoch: {len(train_dl)}')

## 5. Load CatVTON UNet + VAE + tokenizer

CatVTON's UNet expects **12 input channels**: noise (4) + person latent (4) + garment latent (4).

In [ ]:
print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(cfg.base_model, subfolder='vae', torch_dtype=dtype).to(device)
vae.requires_grad_(False)
vae.eval()

print('Loading tokenizer + text encoder...')
tokenizer = CLIPTokenizer.from_pretrained(cfg.base_model, subfolder='tokenizer')
text_encoder = CLIPTextModel.from_pretrained(cfg.base_model, subfolder='text_encoder',
                                              torch_dtype=dtype).to(device)
text_encoder.requires_grad_(False)
text_encoder.eval()

# Empty-prompt embedding (CatVTON conditions on garment latent, not text)
with torch.no_grad():
    null_ids = tokenizer([''], padding='max_length', max_length=77,
                          truncation=True, return_tensors='pt').input_ids.to(device)
    null_emb = text_encoder(null_ids)[0]

print('Loading UNet (12-channel input)...')
unet = UNet2DConditionModel.from_pretrained(
    cfg.base_model, subfolder='unet',
    in_channels=12, ignore_mismatched_sizes=True,
    torch_dtype=dtype,
).to(device)
unet.requires_grad_(False)

print('Loading DDIM scheduler...')
scheduler = DDIMScheduler.from_pretrained(cfg.base_model, subfolder='scheduler')
print(f'Train timesteps: {scheduler.config.num_train_timesteps}')

## 6. Attach LoRA to UNet attention layers

In [ ]:
lora_cfg = LoraConfig(
    r=cfg.lora_rank,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=cfg.target_modules,
    bias='none',
)
unet = get_peft_model(unet, lora_cfg)
unet.print_trainable_parameters()

# Cast LoRA params to fp32 for stable training, base stays fp16
for n, p in unet.named_parameters():
    if p.requires_grad:
        p.data = p.data.float()

## 7. Training loop

In [ ]:
def encode_to_latent(img: torch.Tensor) -> torch.Tensor:
    """img: B,3,H,W in [-1,1] → B,4,h,w latent."""
    with torch.no_grad():
        lat = vae.encode(img.to(dtype)).latent_dist.sample() * vae.config.scaling_factor
    return lat

trainable_params = [p for p in unet.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=cfg.lr, betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8)

total_steps = (len(train_dl) // cfg.grad_accum) * cfg.epochs
def lr_at(step):
    if step < cfg.warmup_steps:
        return step / cfg.warmup_steps
    # Cosine decay
    progress = (step - cfg.warmup_steps) / max(1, total_steps - cfg.warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

scaler = torch.cuda.amp.GradScaler()

global_step = 0
unet.train()

for epoch in range(cfg.epochs):
    pbar = tqdm(train_dl, desc=f'Epoch {epoch+1}/{cfg.epochs}')
    accum_loss = 0.0
    optimizer.zero_grad()
    
    for step, batch in enumerate(pbar):
        person   = batch['person'].to(device)
        agnostic = batch['agnostic'].to(device)
        cloth    = batch['cloth'].to(device)
        mask     = batch['mask'].to(device)
        B = person.shape[0]
        
        # Encode all three to latent space
        person_lat   = encode_to_latent(person)
        agnostic_lat = encode_to_latent(agnostic)
        cloth_lat    = encode_to_latent(cloth)
        
        # Sample noise + timestep
        noise = torch.randn_like(person_lat)
        t = torch.randint(0, scheduler.config.num_train_timesteps, (B,), device=device).long()
        noisy_person_lat = scheduler.add_noise(person_lat, noise, t)
        
        # CatVTON input: [noisy_person_lat(4), agnostic_lat(4), cloth_lat(4)] → 12 channels
        unet_in = torch.cat([noisy_person_lat, agnostic_lat, cloth_lat], dim=1)
        
        # Predict noise
        enc_hidden = null_emb.expand(B, -1, -1)
        with torch.cuda.amp.autocast(dtype=dtype):
            noise_pred = unet(unet_in, t, encoder_hidden_states=enc_hidden).sample
        
        # Masked MSE — only train on the garment region
        # Downsample mask to latent resolution
        mask_lat = F.interpolate(mask, size=noise_pred.shape[-2:], mode='nearest')
        loss = F.mse_loss(noise_pred.float() * mask_lat, noise.float() * mask_lat, reduction='sum')
        loss = loss / (mask_lat.sum() * noise_pred.shape[1] + 1e-6)
        loss = loss / cfg.grad_accum
        
        scaler.scale(loss).backward()
        accum_loss += loss.item() * cfg.grad_accum
        
        if (step + 1) % cfg.grad_accum == 0:
            # LR schedule
            for pg in optimizer.param_groups:
                pg['lr'] = cfg.lr * lr_at(global_step)
            
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, cfg.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            
            global_step += 1
            avg = accum_loss / cfg.grad_accum
            accum_loss = 0.0
            pbar.set_postfix(loss=f'{avg:.4f}', lr=f'{optimizer.param_groups[0]["lr"]:.2e}', step=global_step)
            
            # Checkpoint
            if global_step % cfg.save_every == 0:
                ckpt = Path(cfg.output_dir) / f'step_{global_step}'
                unet.save_pretrained(str(ckpt))
                print(f'  → saved {ckpt}')

# Final save
final = Path(cfg.output_dir) / 'final'
unet.save_pretrained(str(final))
print(f'Done. Final LoRA: {final}')

## 8. Quick sanity inference

Generate one try-on result with the trained LoRA to confirm it works.

In [ ]:
unet.eval()
scheduler.set_timesteps(20)

sample = train_ds[0]
with torch.no_grad():
    person_lat   = encode_to_latent(sample['person'].unsqueeze(0).to(device))
    agnostic_lat = encode_to_latent(sample['agnostic'].unsqueeze(0).to(device))
    cloth_lat    = encode_to_latent(sample['cloth'].unsqueeze(0).to(device))
    
    lat = torch.randn_like(person_lat)
    for t in tqdm(scheduler.timesteps, desc='Denoise'):
        unet_in = torch.cat([lat, agnostic_lat, cloth_lat], dim=1)
        tb = t.unsqueeze(0).to(device)
        with torch.cuda.amp.autocast(dtype=dtype):
            pred = unet(unet_in, tb, encoder_hidden_states=null_emb).sample
        lat = scheduler.step(pred.float(), t, lat).prev_sample
    
    img = vae.decode(lat.to(dtype) / vae.config.scaling_factor).sample
    img = ((img.clamp(-1, 1) + 1) * 127.5).byte().cpu().numpy()[0].transpose(1, 2, 0)
    out = Image.fromarray(img)
    out.save('/kaggle/working/sample_result.jpg')

# Display side-by-side
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow((sample['agnostic'].permute(1,2,0).numpy() + 1) / 2); axes[0].set_title('Input (agnostic)')
axes[1].imshow((sample['cloth'].permute(1,2,0).numpy() + 1) / 2);    axes[1].set_title('Target garment')
axes[2].imshow((sample['person'].permute(1,2,0).numpy() + 1) / 2);   axes[2].set_title('Ground truth')
axes[3].imshow(out);                                                   axes[3].set_title('LoRA prediction')
for a in axes: a.axis('off')
plt.tight_layout(); plt.savefig('/kaggle/working/comparison.jpg', dpi=100); plt.show()

## 9. Zip the LoRA for download

The output zip is what you upload to your RunPod backend.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/lora_catvton_final', 'zip', cfg.output_dir, 'final')
size_mb = os.path.getsize('/kaggle/working/lora_catvton_final.zip') / 1e6
print(f'lora_catvton_final.zip — {size_mb:.1f} MB')
print('Right-click → Download from Kaggle output panel')

## 10. After training: use the LoRA in your backend

On your RunPod/server:

```bash
# 1. Unzip the LoRA into your backend dir
unzip lora_catvton_final.zip -d /workspace/lora_catvton

# 2. Set the env var that tier 2 in your model.py reads
export VTON_LORA_CHECKPOINT=/workspace/lora_catvton

# 3. Restart your tryon_backend server
python server.py
```

Your existing [tryon_backend/model.py](../tryon_backend/model.py) `_load_tier2()` reads `VTON_LORA_CHECKPOINT` and loads it as the UNet. The fine-tuned weights take over automatically.

## Tips for better quality

- **More steps:** raise `epochs` to 2-3. Each is ~6hrs on Kaggle P100.
- **More data:** set `train_subset = 0` to use all ~11k pairs (slower but better).
- **Bigger images:** raise `image_size` to 512. Needs `batch_size=1` on P100, or T4 ×2 with model-sharding.
- **Higher LoRA rank:** `lora_rank=32` learns more but file size grows ~2×.
- **Class-specific:** if you only care about jackets, filter `pairs` by category before training (VITON-HD has parse maps with class labels).